# CMSC 173 &middot; Module 11 &middot; Example 4
## Unusual earthquakes

First Part 7's six numbers, 1, 2, 3, 4, 5 and 20, through every detector by hand; then 11,391 Philippine earthquakes through two of them, Isolation Forest and the local outlier factor.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides; change something and see what moves.

**Part 7: z-score and IQR on six numbers.**

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5, 20.0])
z = (x - x.mean()) / x.std()                          # NumPy's std divides by n
print("mean", x.mean().round(2), "SD", x.std().round(2), "z", z.round(2))   # z of 20: 2.19, under 3
q1, q3 = np.percentile(x, [25, 75])
print("Q1", q1, "Q3", q3, "fences", q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))   # -1.5 and 8.5: 20 flagged
# pandas' .std() divides by n - 1: SD 7.08 and z = 2.0. Still under 3.

**Distance and density.**

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors

col = x.reshape(-1, 1)
print("nearest neighbour:", NearestNeighbors(n_neighbors=2).fit(col).kneighbors(col)[0][:, 1])   # 15 for 20
lof = LocalOutlierFactor(n_neighbors=2).fit(col)
print("LOF:", (-lof.negative_outlier_factor_).round(2))        # 10.33 for 20
print("DBSCAN:", DBSCAN(eps=1, min_samples=2).fit_predict(col))  # -1 = noise

**Isolation by hand.** Cut at a random value between the smallest and largest number; keep the side holding the point; repeat until it is alone. Average over 10,000 random trees.

In [ ]:
rng = np.random.default_rng(173)

def cuts(target, pts):
    n = 0
    while len(pts) > 1:
        c = rng.uniform(pts.min(), pts.max())
        n += 1
        pts = pts[pts <= c] if target <= c else pts[pts > c]
    return n

print([round(float(np.mean([cuts(t, x) for _ in range(10000)])), 2) for t in x])
# [2.86, 3.67, 3.89, 3.78, 3.09, 1.23]: 20 is alone after about one cut

**Review question.** With 200 instead of 20, the z-score still stays under 3: with six numbers it can never pass $\sqrt{5}$.

In [ ]:
v = np.array([1, 2, 3, 4, 5, 200.0])
print(round((200 - v.mean()) / v.std(), 3), round(5 ** 0.5, 3))   # 2.236 2.236

**Example 4: the earthquakes.** USGS catalogue, magnitude 4.5 and above, 2000&ndash;2026.

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

url = "https://raw.githubusercontent.com/njpinton/cmsc173-labs/main/data/ph_earthquakes.csv"
quakes = pd.read_csv(url)
X = quakes[["longitude", "latitude", "depth"]]
iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=173).fit(X)
iso_flag = iso.predict(X) == -1                     # -1 means anomaly
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
lof_flag = lof.fit_predict(StandardScaler().fit_transform(X)) == -1   # distances: scale first
print(iso_flag.sum(), lof_flag.sum(), (iso_flag & lof_flag).sum())   # 113 114 8

**What each detector flagged.**

In [ ]:
print("Isolation Forest, depth (km):", quakes.depth[iso_flag].min(), "to", quakes.depth[iso_flag].max())   # 500.0 to 678.0
print("LOF, median depth (km):", quakes.depth[lof_flag].median())                                        # 10.0
z_depth = (quakes.depth - quakes.depth.mean()) / quakes.depth.std()
print("depth z > 3:", (z_depth.abs() > 3).sum(), "| of them flagged by Isolation Forest:",
      ((z_depth.abs() > 3) & iso_flag).sum())                                                           # 265 | 113

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
for ax, flag, name, colour in [(axes[0], iso_flag, "Isolation Forest", "red"), (axes[1], lof_flag, "LOF", "purple")]:
    ax.scatter(quakes.longitude, quakes.latitude, s=1, c="lightgrey")
    ax.scatter(quakes.longitude[flag], quakes.latitude[flag], s=12, c=colour)
    ax.set_title(f"{name}: {flag.sum()} flagged"); ax.set_xlabel("longitude"); ax.set_aspect("equal")
axes[0].set_ylabel("latitude")
plt.show()

**Try it.** Add `mag` to `X`. Which quakes does Isolation Forest flag now? Then give LOF `n_neighbors=5`: does it agree with Isolation Forest more, or less?